In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Dataset 

In [5]:
file_path = "Merged_TSQIC_REDCap.xlsx" 
merged_final_tsqic_redcap_access = pd.read_excel(file_path)
merged_final_tsqic_redcap_access

,id,operation_date,Complication,Grade,GradeLetter,readmission_30d,DischargeDate,los,redcap_event_name,redcap_repeat_instrument,redcap_repeat_instance,path_proximalmargin,path_distalmargin,path_circummargin,path_lymph_total,intraop_complications
0,1,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0
1,1,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,1.0,1.0,1.0,1.0,34.0,NaN
2,2,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,1.0,1.0,1.0,1.0,41.0,NaN
3,2,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0
4,2,NaT,NaN,NaN,NaN,NaN,NaT,NaN,1_month_postop_arm_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5408,1764,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,1.0,1.0,1.0,2.0,46.0,NaN
5409,1766,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0
5410,1766,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgical_pathology,2.0,1.0,1.0,1.0,50.0,NaN
5411,1770,NaT,NaN,NaN,NaN,NaN,NaT,NaN,surgery_arm_1,surgery_esd_emr,1.0,NaN,NaN,NaN,NaN,0.0


In [6]:
len(merged_final_tsqic_redcap_access['id'].unique())

1270

# Analysis

In [7]:
# Define the criteria
criteria = [
    'path_proximalmargin',
    'path_distalmargin', 
    'path_circummargin',
    'path_lymph_total',
    'intraop_complications',
    'Grade',
    'Complication',
    'readmission_30d',
    'los'
]

# Group data by patient ID
patient_groups = merged_final_tsqic_redcap_access.groupby('id')

# Track results for each patient
patient_outcomes = {}
failed_criteria_counts = {
    'path_proximalmargin': 0,
    'path_distalmargin': 0,
    'path_circummargin': 0,
    'path_lymph_total': 0,
    'intraop_complications': 0,
    'Grade': 0,
    'Complication_not_leak': 0,
    'readmission_30d': 0,
    'los': 0
}
patient_failed_criteria = {}

In [8]:
# Process each patient
for patient_id, patient_data in patient_groups:
    # Initialize tracking for this patient
    criteria_met = {
        'path_proximalmargin': False,
        'path_distalmargin': False,
        'path_circummargin': False,
        'path_lymph_total': False,
        'intraop_complications': False,
        'Grade': False,
        'Complication_not_leak': False,
        'readmission_30d': False,
        'los': False
    }
    criteria_failed = {
        'path_proximalmargin': False,
        'path_distalmargin': False,
        'path_circummargin': False,
        'path_lymph_total': False,
        'intraop_complications': False,
        'Grade': False,
        'Complication_not_leak': False,
        'readmission_30d': False,
        'los': False
    }
    
    # Check all rows for this patient
    for _, row in patient_data.iterrows():
        # Check each criterion
        if pd.notna(row['path_proximalmargin']):
            if row['path_proximalmargin'] == 1:
                criteria_met['path_proximalmargin'] = True
            else:
                criteria_failed['path_proximalmargin'] = True
                
        if pd.notna(row['path_distalmargin']):
            if row['path_distalmargin'] == 1:
                criteria_met['path_distalmargin'] = True
            else:
                criteria_failed['path_distalmargin'] = True
                
        if pd.notna(row['path_circummargin']):
            if row['path_circummargin'] == 1:
                criteria_met['path_circummargin'] = True
            else:
                criteria_failed['path_circummargin'] = True
                
        if pd.notna(row['path_lymph_total']):
            if row['path_lymph_total'] >= 20:
                criteria_met['path_lymph_total'] = True
            else:
                criteria_failed['path_lymph_total'] = True
                
        if pd.notna(row['intraop_complications']):
            if row['intraop_complications'] == 0:
                criteria_met['intraop_complications'] = True
            else:
                criteria_failed['intraop_complications'] = True
                
        if pd.notna(row['Grade']):
            if row['Grade'] < 3:
                criteria_met['Grade'] = True
            else:
                criteria_failed['Grade'] = True
                
        if pd.notna(row['Complication']):
            if row['Complication'] != 'Leak':
                criteria_met['Complication_not_leak'] = True
            else:
                criteria_failed['Complication_not_leak'] = True
                
        if pd.notna(row['readmission_30d']):
            if row['readmission_30d'] == 0:
                criteria_met['readmission_30d'] = True
            else:
                criteria_failed['readmission_30d'] = True
                
        if pd.notna(row['los']):
            if row['los'] < 14:
                criteria_met['los'] = True
            else:
                criteria_failed['los'] = True
    
    # Determine if this patient achieves textbook outcome
    all_criteria_have_data = all(criteria_met.values())
    no_criteria_failed = not any(criteria_failed.values())
    
    textbook_outcome = all_criteria_have_data and no_criteria_failed
    patient_outcomes[patient_id] = textbook_outcome
    
    # If patient fails, record which criteria they failed on
    if not textbook_outcome:
        patient_failed_criteria[patient_id] = {}
        # Failed due to missing data
        for criterion, met in criteria_met.items():
            if not met:
                failed_criteria_counts[criterion] += 1
                patient_failed_criteria[patient_id][criterion] = "missing_data"
        
        # Failed due to not meeting criterion
        for criterion, failed in criteria_failed.items():
            if failed:
                failed_criteria_counts[criterion] += 1
                patient_failed_criteria[patient_id][criterion] = "criterion_failed"


In [11]:
patient_failed_criteria

{1: {'Grade': 'missing_data',
  'Complication_not_leak': 'missing_data',
  'readmission_30d': 'missing_data',
  'los': 'missing_data'},
 2: {'Grade': 'missing_data',
  'Complication_not_leak': 'missing_data',
  'readmission_30d': 'missing_data',
  'los': 'missing_data'},
 3: {'path_proximalmargin': 'criterion_failed',
  'path_distalmargin': 'missing_data',
  'path_lymph_total': 'criterion_failed',
  'intraop_complications': 'criterion_failed',
  'Grade': 'missing_data',
  'Complication_not_leak': 'missing_data',
  'readmission_30d': 'missing_data',
  'los': 'missing_data'},
 4: {'path_circummargin': 'criterion_failed',
  'path_lymph_total': 'criterion_failed',
  'intraop_complications': 'criterion_failed',
  'Grade': 'missing_data',
  'Complication_not_leak': 'missing_data',
  'readmission_30d': 'missing_data',
  'los': 'missing_data'},
 5: {'path_circummargin': 'criterion_failed',
  'intraop_complications': 'criterion_failed',
  'Grade': 'missing_data',
  'Complication_not_leak': 'mis

In [12]:
patient_outcomes

{1: False,
 2: False,
 3: False,
 4: False,
 5: False,
 6: False,
 7: False,
 8: False,
 9: False,
 10: False,
 12: False,
 13: False,
 15: False,
 17: False,
 18: False,
 19: False,
 20: False,
 21: False,
 22: False,
 23: False,
 24: False,
 25: False,
 26: False,
 27: False,
 28: False,
 29: False,
 30: False,
 31: False,
 32: False,
 33: False,
 34: False,
 35: False,
 36: False,
 37: False,
 38: False,
 39: False,
 40: False,
 41: False,
 42: False,
 43: False,
 44: False,
 47: False,
 48: False,
 49: False,
 50: False,
 51: False,
 54: False,
 55: False,
 57: False,
 58: False,
 59: False,
 60: False,
 61: False,
 62: False,
 63: False,
 64: False,
 65: False,
 67: False,
 68: False,
 69: False,
 70: False,
 72: False,
 73: False,
 74: False,
 75: False,
 76: False,
 77: False,
 78: False,
 79: False,
 81: False,
 82: False,
 83: False,
 84: False,
 85: False,
 86: False,
 88: False,
 89: False,
 90: False,
 91: False,
 92: False,
 93: False,
 94: False,
 95: False,
 96: False,
 

In [14]:
# Calculate overall textbook outcome percentage
total_patients = len(patient_outcomes)
textbook_outcome_count = sum(1 for outcome in patient_outcomes.values() if outcome)
textbook_outcome_percentage = (textbook_outcome_count / total_patients) * 100 if total_patients > 0 else 0
print(total_patients)
print(textbook_outcome_count)
print(textbook_outcome_percentage)

1270
71
5.590551181102362


In [15]:
# Calculate percentages for failed criteria
failed_patient_count = total_patients - textbook_outcome_count
failed_criteria_percentages = {
    criterion: (count / failed_patient_count) * 100 
    for criterion, count in failed_criteria_counts.items()
} if failed_patient_count > 0 else {criterion: 0 for criterion in failed_criteria_counts}
print(failed_criteria_percentages)

{'path_proximalmargin': 22.768974145120936, 'path_distalmargin': 19.849874895746456, 'path_circummargin': 47.706422018348626, 'path_lymph_total': 62.8023352793995, 'intraop_complications': 45.45454545454545, 'Grade': 97.08090075062552, 'Complication_not_leak': 80.40033361134279, 'readmission_30d': 75.39616346955796, 'los': 77.3978315262719}


In [16]:
# Generate summary statistics
print(f"Total patients included in analysis: {total_patients}")
print(f"Patients with textbook outcome: {textbook_outcome_count} ({textbook_outcome_percentage:.2f}%)")
print(f"Patients without textbook outcome: {failed_patient_count} ({100 - textbook_outcome_percentage:.2f}%)")
print("\nBreakdown of failed criteria (among patients without textbook outcome):")
for criterion, count in failed_criteria_counts.items():
    percentage = failed_criteria_percentages[criterion]
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")

Total patients included in analysis: 1270
Patients with textbook outcome: 71 (5.59%)
Patients without textbook outcome: 1199 (94.41%)

Breakdown of failed criteria (among patients without textbook outcome):
  path_proximalmargin: 273 patients (22.77%)
  path_distalmargin: 238 patients (19.85%)
  path_circummargin: 572 patients (47.71%)
  path_lymph_total: 753 patients (62.80%)
  intraop_complications: 545 patients (45.45%)
  Grade: 1164 patients (97.08%)
  Complication_not_leak: 964 patients (80.40%)
  readmission_30d: 904 patients (75.40%)
  los: 928 patients (77.40%)


In [17]:
# Sort criteria by difficulty (most failed to least failed)
sorted_criteria = sorted(
    failed_criteria_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

print("\nCriteria ranked by difficulty (most failed to least failed):")
for criterion, count in sorted_criteria:
    percentage = failed_criteria_percentages[criterion]
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")


Criteria ranked by difficulty (most failed to least failed):
  Grade: 1164 patients (97.08%)
  Complication_not_leak: 964 patients (80.40%)
  los: 928 patients (77.40%)
  readmission_30d: 904 patients (75.40%)
  path_lymph_total: 753 patients (62.80%)
  path_circummargin: 572 patients (47.71%)
  intraop_complications: 545 patients (45.45%)
  path_proximalmargin: 273 patients (22.77%)
  path_distalmargin: 238 patients (19.85%)


In [18]:
# Additional analysis: Distinguish between missing data and failed criteria
missing_data_counts = {criterion: 0 for criterion in criteria_met.keys()}
failed_criteria_only_counts = {criterion: 0 for criterion in criteria_met.keys()}

for patient_id, failures in patient_failed_criteria.items():
    for criterion, reason in failures.items():
        if reason == "missing_data":
            missing_data_counts[criterion] += 1
        elif reason == "criterion_failed":
            failed_criteria_only_counts[criterion] += 1

print("\nBreakdown of failures by reason:")
print("Missing data:")
for criterion, count in missing_data_counts.items():
    percentage = (count / failed_patient_count) * 100 if failed_patient_count > 0 else 0
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")

print("\nFailed criteria (data present but criterion not met):")
for criterion, count in failed_criteria_only_counts.items():
    percentage = (count / failed_patient_count) * 100 if failed_patient_count > 0 else 0
    print(f"  {criterion}: {count} patients ({percentage:.2f}%)")


Breakdown of failures by reason:
Missing data:
  path_proximalmargin: 159 patients (13.26%)
  path_distalmargin: 162 patients (13.51%)
  path_circummargin: 189 patients (15.76%)
  path_lymph_total: 142 patients (11.84%)
  intraop_complications: 343 patients (28.61%)
  Grade: 840 patients (70.06%)
  Complication_not_leak: 840 patients (70.06%)
  readmission_30d: 840 patients (70.06%)
  los: 605 patients (50.46%)

Failed criteria (data present but criterion not met):
  path_proximalmargin: 59 patients (4.92%)
  path_distalmargin: 40 patients (3.34%)
  path_circummargin: 195 patients (16.26%)
  path_lymph_total: 314 patients (26.19%)
  intraop_complications: 101 patients (8.42%)
  Grade: 268 patients (22.35%)
  Complication_not_leak: 117 patients (9.76%)
  readmission_30d: 53 patients (4.42%)
  los: 193 patients (16.10%)
